# Ingest new histology

Takes one animal's whole-slide scans to reviewed GFP coverage per target:

1. **Export** each `.vsi` to per-section OME-TIFFs (skips sections already exported)
2. **Orient**: in napari, click the brain's **centre**, its **front**, and the **side the notch is on** for every section, then press **Done**
3. **Save** the clicks and check that none are missing or ambiguous
4. **Finetune** (optional): in napari, drag, resize or rotate each of T1–T6 on its own, then press **Done**. Saved per target in `data/roi_locations.csv`
5. **Measure** GFP coverage in each target's ROI: the finetuned shape if saved, otherwise the template placed from your clicks. There is no automatic GFP fitting
6. **Review**: in napari, check the measured ROIs and coverage on each section, mark it ✓ or ✗, then press **Done**
7. **Result**: coverage per target from the sections that passed

Clicks and review verdicts are saved in `data/section_orientation.csv` (tracked in git). They are saved on every move between sections, so closing a window loses nothing.

Why the clicks are needed: `docs/histology-pipeline.md#mouse_02`. Kernel: **Python 3.12 (fus-research .venv)**.

In [1]:
%gui qt
from pathlib import Path

import numpy as np

import pandas as pd

from fus import histology, orientation, rois

ROOT = histology.REPO_ROOT

# ---- parameters: set these for each new animal ------------------------------
MOUSE = 2
VSI = sorted((ROOT / f"data/histology/Mouse_{MOUSE:02d}").glob("*.vsi"))
OUT = ROOT / f"data/processed/histology/Mouse_{MOUSE:02d}/ds4"
# Template runs get their own files: mouse01_slots.csv is the GFP-fit benchmark.
SLOTS_CSV = ROOT / f"results/histology/mouse{MOUSE:02d}_template_slots.csv"
FIGURES = ROOT / f"results/histology/figures/mouse{MOUSE:02d}_template"

# Mouse 1's notch clicks tell the pipeline which side of the animal targets 1-3
# are on. They only need doing once; leave False once they're in the sheet.
INCLUDE_MOUSE01 = False
MOUSE01 = sorted((ROOT / "data/processed/histology/Mouse_01/ds4").glob("section_s*.ome.tif"))

print(f"Mouse {MOUSE}: {len(VSI)} scan(s)")
for v in VSI:
    print("  ", v.relative_to(ROOT))

Mouse 2: 4 scan(s)
   data/histology/Mouse_02/Control_02_Slide_01.vsi
   data/histology/Mouse_02/Control_02_Slide_02.vsi
   data/histology/Mouse_02/Control_02_Slide_03.vsi
   data/histology/Mouse_02/Control_02_Slide_04.vsi


## 1. Export

Olympus scans hold a label image, an overview, and one series per section. Each section is written at 4× downsampling (1.3 µm/px), about 20 s each. Mouse 1 was a single `Image.vsi`; later animals come as one `.vsi` per slide, so each slide's sections get a `slideNN_` prefix.

In [2]:
import re


def prefix_for(vsi: Path) -> str:
    m = re.search(r"Slide_?0*(\d+)", vsi.stem)
    return f"slide{int(m.group(1)):02d}" if m else "section"


for vsi in VSI:
    prefix = prefix_for(vsi)
    if list(OUT.glob(f"{prefix}_s*.ome.tif")):
        print(f"{vsi.name}: already exported, skipping")
        continue
    for path in histology.export_sections(vsi, OUT, prefix=prefix):
        print(f"{vsi.name}: wrote {path.name}")

SECTIONS = sorted(OUT.glob("*.ome.tif"))
print(f"\n{len(SECTIONS)} sections:", ", ".join(orientation.section_name(p) for p in SECTIONS))

Control_02_Slide_01.vsi: already exported, skipping
Control_02_Slide_02.vsi: already exported, skipping
Control_02_Slide_03.vsi: already exported, skipping
Control_02_Slide_04.vsi: already exported, skipping

12 sections: slide01_s2, slide01_s3, slide01_s4, slide02_s2, slide02_s3, slide02_s4, slide03_s2, slide03_s3, slide03_s4, slide04_s2, slide04_s3, slide04_s4


## 2. Orient — napari

A window opens with each section in turn: NeuN in grey, the GFP stain in green. For each section, click in order:

1. **centre** (cyan): the middle of the brain, on the midline
2. **front** (red): the front tip of the brain, opposite the cerebellum
3. **notch side** (yellow): out to the side the notch is on, roughly at a right angle to the front arrow

After each click the next dot is selected. Arrows run from the centre, and the **yellow circles are the target template**. It's a rigid copy of the six-target plan at a fixed size, sitting ~1.6 mm in front of your centre and turned to your front arrow. Move the centre or front dot until it sits on the targets. Labels become T1–T6 once the notch side is set (Mouse 1's notch clicks are needed for that). **Space** saves and moves on. Tick **Exclude** for sections with no usable GFP, or that are torn or partial. When you've finished, press **Done** to save and close the window, then run the next cell.

Sections already clicked are loaded, so you can check or adjust them. Clicked sections are shown **straightened**: front up, notch side left, mirror-flipped if the section lies face down. This is a display transform only; the pixels are not resampled. New sections stay as scanned so the view doesn't turn while you click. Press **R** to straighten after clicking. The finetune and review windows are always straightened.

In [ ]:
to_orient = SECTIONS + (MOUSE01 if INCLUDE_MOUSE01 else [])
annotator = orientation.annotate(to_orient)

## 3. Save the selections

Reads back what was clicked, flags anything missing or ambiguous, and records the table in this notebook.

In [3]:
sheet = orientation.read_sheet()
names = [orientation.section_name(p) for p in SECTIONS]

rows = []
for name in names:
    a = sheet.get((MOUSE, name), orientation.Annotation(MOUSE, name))
    rows.append({
        "section": name,
        "front points": orientation._compass(a.angle_deg) if a.angle_deg is not None else "",
        "front angle": None if a.angle_deg is None else round(a.angle_deg),
        "notch arrow vs front": None if a.notch_offset_deg is None else f"{a.notch_offset_deg:.0f}°",
        "exclude": a.exclude,
        "complete": a.complete,
        "notes": a.notes,
    })
selections = pd.DataFrame(rows).set_index("section")
display(selections)

missing = selections.index[~selections.complete].tolist()
ambiguous = [n for n in names if (o := sheet.get((MOUSE, n))) and o.notch_offset_deg is not None
             and abs(o.notch_offset_deg - 90) > 60]
plus_x = orientation.plus_x_side(sheet)
print("missing clicks:", ", ".join(missing) or "none")
print("notch-side arrow too close to the midline:", ", ".join(ambiguous) or "none")
print("targets 1-3 are on the animal's", plus_x or "?? (click the Mouse 1 sections: INCLUDE_MOUSE01 = True)")
assert not missing and not ambiguous, "fix these in the annotator (re-run the cell above) before measuring"

,front points,front angle,notch arrow vs front,exclude,complete,notes
section,,,,,,
slide01_s2,up,197,89°,False,True,typed 2026-09-30: notch bottom-right; automati...
slide01_s3,up,183,92°,False,True,typed 2026-09-30: notch bottom-right; review o...
slide01_s4,up,165,93°,False,True,typed 2026-09-30: notch top-left; little GFP; ...
slide02_s2,up,175,91°,False,True,typed 2026-09-30: notch bottom-left; automatic...
slide02_s3,up,193,90°,False,True,typed 2026-09-30: notch bottom-left; review of...
slide02_s4,up,190,89°,False,True,typed 2026-09-30: notch bottom-left; notch har...
slide03_s2,up-right,210,89°,False,True,typed 2026-09-30: notch bottom-left; slot 5 mo...
slide03_s3,up-left,134,86°,False,True,typed 2026-09-30: notch bottom-left; automatic...
slide03_s4,up,189,87°,False,True,little GFP; review of old GFP-fit placement: w...


missing clicks: none
notch-side arrow too close to the midline: none
targets 1-3 are on the animal's right


## 4. Finetune — napari (optional)

Each section opens with T1–T6 as six yellow shapes, starting from the template (or from your last saved shapes). With the **targets** layer selected and in select mode (**S**):

- click a shape and drag it to **move** it
- drag a corner handle to **resize** (hold **Shift** to keep it round)
- drag the handle above the box to **rotate**
- hold **Space** and drag to pan

The section is shown front up, notch side left, like the orientation window.

Moving between sections saves the one you leave. **Reset section to template** starts a section over. Press **Done** when finished; skip this step to measure with the template.

Two cautions:
- **Resizing changes the measurement.** Coverage is the GFP+ fraction *inside the shape*, so a smaller shape on a bright plume reads higher. Keep the 2 mm circles unless there's an anatomical reason to change them.
- **Placing shapes by eye on the GFP** can pull a target onto its own signal. Anatomy is the safer guide.

In [4]:
finetuner = rois.finetune([p for p in SECTIONS
                           if not sheet[(MOUSE, orientation.section_name(p))].exclude])

### Save the finetuned locations

The finetune window already wrote `data/roi_locations.csv`. This reads it back and records it here, with how far each target moved from the template and its size.

In [5]:
sheet = orientation.read_sheet()
plus_x = orientation.plus_x_side(sheet)
locations = rois.read_locations()
rows = []
for path in SECTIONS:
    name = orientation.section_name(path)
    saved = locations.get((MOUSE, name))
    a = sheet[(MOUSE, name)]
    if saved is None or a.exclude:
        continue
    px_um = rois.export_pixel_um(path)
    base = rois.from_template(a, px_um, orientation.is_mirrored(a, plus_x))
    for t, roi in sorted(saved.items()):
        rows.append({"section": name, "target": f"T{t}",
                     "moved_mm": round(float(np.hypot(*np.subtract(roi.centre, base[t].centre))) * px_um / 1000, 2),
                     "size_mm": f"{2 * roi.semi_a * px_um / 1000:.2f} × {2 * roi.semi_b * px_um / 1000:.2f}",
                     "centre_row": round(roi.centre[0]), "centre_col": round(roi.centre[1])})
finetuned = pd.DataFrame(rows)
print(f"{finetuned.section.nunique() if len(finetuned) else 0} of {len(SECTIONS)} sections finetuned")
display(finetuned.set_index(["section", "target"]) if len(finetuned) else finetuned)

12 of 12 sections finetuned


moved_mm      size_mm  centre_row  centre_col
section    target                                               
slide01_s2 T1          0.08  1.74 × 1.74        2002        3423
           T2          0.17  1.74 × 1.74        2755        1670
           T3          0.10  1.74 × 1.74        3618        2953
           T4          0.11  1.74 × 1.74        2560        5374
           T5          0.07  1.74 × 1.74        4189        6230
...                     ...          ...         ...         ...
slide04_s4 T2          0.00  1.74 × 1.74        3598        6851
           T3          0.00  1.74 × 1.74        3751        5480
           T4          0.00  1.74 × 1.74        1825        3714
           T5          0.00  1.74 × 1.74        2973        2208
           T6          0.00  1.74 × 1.74        3483        3490

[72 rows x 4 columns]

## 5. Measure

GFP coverage in each target's ROI. That's the finetuned shape where one was saved in step 4, otherwise the 1.0 mm template circle exactly where the orientation window drew it. Nothing is fitted to the GFP automatically. It takes about 12 s per section.

In [6]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGURES.mkdir(parents=True, exist_ok=True)
locations = rois.read_locations()
tables = []
for path in SECTIONS:
    a = sheet[(MOUSE, orientation.section_name(path))]
    if a.exclude:
        print(f"{a.section}: excluded")
        continue
    sec = histology.load_section(path)
    template = histology.template_fit(a.centre, a.angle_deg, sec.pixel_um)
    finetuned = locations.get((MOUSE, sec.name))
    res = histology.analyse_section(sec, template=template, rois=finetuned)
    tables.append(res.table())
    fig = histology.plot_section(sec, res)
    fig.savefig(FIGURES / f"{sec.name}_slots.png", dpi=110)
    plt.close(fig)
    cov = {r["slot"]: r["coverage"] for r in res.rows if r["channel"] == "TRITC"}
    how = "finetuned" if finetuned else "template "
    print(f"{sec.name}: {how}  TRITC by slot " + " ".join(f"{cov[s]:.2f}" for s in sorted(cov)))
    del sec, res

slots = pd.concat(tables, ignore_index=True)
slots.to_csv(SLOTS_CSV, index=False)
print(f"\nwrote {len(slots)} rows to {SLOTS_CSV.relative_to(ROOT)}")

slide01_s2: finetuned  TRITC by slot 0.09 0.19 0.06 0.13 0.02 0.01
slide01_s3: finetuned  TRITC by slot 0.50 0.25 0.88 0.88 0.16 0.87
slide01_s4: finetuned  TRITC by slot 0.26 0.32 0.37 0.08 0.02 0.12
slide02_s2: finetuned  TRITC by slot 0.11 0.04 0.02 0.05 0.46 0.14
slide02_s3: finetuned  TRITC by slot 0.63 0.14 0.94 0.07 0.03 0.43
slide02_s4: finetuned  TRITC by slot 0.81 0.14 0.86 0.43 0.25 0.89
slide03_s2: finetuned  TRITC by slot 0.83 0.33 0.94 0.05 0.06 0.68
slide03_s3: finetuned  TRITC by slot 0.48 0.20 0.65 0.01 0.01 0.17
slide03_s4: finetuned  TRITC by slot 0.02 0.01 0.01 0.10 0.51 0.08
slide04_s2: finetuned  TRITC by slot 0.07 0.41 0.05 0.11 0.07 0.01
slide04_s3: finetuned  TRITC by slot 0.42 0.15 0.86 0.80 0.17 0.87
slide04_s4: finetuned  TRITC by slot 0.14 0.02 0.16 0.33 0.42 0.45

wrote 144 rows to results/histology/mouse02_template_slots.csv


## 6. Review — napari

A window shows each measured section with its ROIs exactly as measured (finetuned shapes or template circles), labelled **T1–T6** with TRITC coverage, plus your arrows. For each section, check that each ROI sits on its target and the labels are on the right side, then press:

- **Looks right ✓** (Y), or
- **Wrong ✗** (X). Wrong sections are left out of the result.

Press **Done** when finished, then run the last cell.

In [7]:
reviewer = orientation.review([p for p in SECTIONS
                               if not sheet[(MOUSE, orientation.section_name(p))].exclude],
                              pd.read_csv(SLOTS_CSV))

## 7. Coverage per target

This uses only sections that are clicked, not excluded, and marked ✓ in the review. Values are TRITC (anti-GFP stain) coverage in the 1.0 mm ROI. Joining these to the acoustic dose is `scripts/mouse02_dose_vs_coverage.py` (for Mouse 2).

In [8]:
sheet = orientation.read_sheet()
plus_x = orientation.plus_x_side(sheet)
slots = pd.read_csv(SLOTS_CSV)

verdicts = pd.Series({n: sheet[(MOUSE, n)].review or "not reviewed" for n in names}, name="review")
display(verdicts.to_frame())

good = [n for n in names
        if sheet[(MOUSE, n)].review == "ok"
        and not slots.loc[slots.section == n, "angle_flag"].any()]
d = slots[slots.section.isin(good) & (slots.channel == "TRITC")].copy()
d["target"] = [histology.slot_to_target(s, orientation.is_mirrored(sheet[(MOUSE, n)], plus_x))
               for s, n in zip(d.slot, d.section)]

per_section = d.pivot(index="section", columns="target", values="coverage").round(2)
print(f"{len(good)} of {len(names)} sections used")
display(per_section)
summary = d.groupby("target").coverage.agg(["mean", "min", "max", "count"]).round(2)
display(summary)

,review
slide01_s2,not reviewed
slide01_s3,not reviewed
slide01_s4,not reviewed
slide02_s2,not reviewed
slide02_s3,not reviewed
slide02_s4,not reviewed
slide03_s2,not reviewed
slide03_s3,not reviewed
slide03_s4,not reviewed
slide04_s2,not reviewed


0 of 12 sections used


target
section


,mean,min,max,count
target,,,,
